## Eem

In [0]:

# 0) IMPORTS & SPARK SESSION

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, expr, concat, lit,
    regexp_replace, floor, udf, date_format,
    when
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("MergeGroundTruthOCR").getOrCreate()


# 1) TUNE SHUFFLE & PARALLELISM

spark.conf.set("spark.sql.shuffle.partitions", "12")


# 2) PATH DEFINITIONS

gt_path      = "xxx/Eem/GroundTruth/AllGroundTruth_Eem_Ch04_20240619_060343_20240619_235951.csv"
ocr_dir      = "xxx/Eem/metadata/metadata.csv"
decoded_root = "xxx/Eem/Decoded Frames/Eem_ch04_0619_060343_235956"
embed_root   = "xxx/Eem/Embeddings/Eem_ch04_0619_060343_235956"
output_path  = "xxx/Eem/Final Metadata/Final_metadata_Eem_Ch04_20240619_060343_20240619_235951.csv"


# 3) DRIVER-SIDE: COLLECT VALID PATHS & BROADCAST

valid_frames = []
for video_dir in [d.name.rstrip("/") for d in dbutils.fs.ls(decoded_root) if d.isDir()]:
    for f in dbutils.fs.ls(f"{decoded_root}/{video_dir}"):
        if f.name.lower().endswith(".jpg"):
            valid_frames.append(f.path)

valid_embeds = []
for obj_dir in [d.name.rstrip("/") for d in dbutils.fs.ls(embed_root) if d.isDir()]:
    for f in dbutils.fs.ls(f"{embed_root}/{obj_dir}"):
        if f.name.lower().endswith(".pt"):
            valid_embeds.append(f.path)

bc_frames = spark.sparkContext.broadcast(set(valid_frames))
bc_embeds = spark.sparkContext.broadcast(set(valid_embeds))

@udf(StringType())
def validate_frame(path_str):
    return path_str if path_str in bc_frames.value else None

@udf(StringType())
def validate_embed(path_str):
    return path_str if path_str in bc_embeds.value else None


# 4) READ & PARSE GROUND-TRUTH CSV

gt = (
    spark.read
         .option("header", True)
         .csv(gt_path)
         .withColumn(
             "gt_ts",
             to_timestamp(col("Timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX")
         )
)


# 5) READ & PARSE OCR CSV PARTS, FILTER "Success"

ocr = (
    spark.read
         .option("header", True)
         .csv(f"{ocr_dir}/*.csv")
         .filter(col("status") == "Success")
         .withColumn(
             "ocr_ts",
             to_timestamp(col("actual_timestamp"), "dd/MM/yyyy HH:mm:ss")
         )
)


# 6) INNER JOIN ON 1s WINDOW & EXTRACT SECOND-TS

joined = gt.join(
    ocr,
    (col("gt_ts") >= col("ocr_ts")) &
    (col("gt_ts") < expr("ocr_ts + interval 1 second")),
    how="inner"
).withColumn("sec_ts", col("ocr_ts"))


# 7) COMPUTE FRAME_NUM, PATHS

joined = (
    joined
    .withColumn("frame_num", regexp_replace(col("frame_name"), "\\D+", "").cast("int"))
    .withColumn("video_index", floor((col("frame_num") - 1) / 3000) + 1)
    .withColumn(
        "Frame Directory",
        concat(
            lit(f"{decoded_root}/video"),
            col("video_index").cast("string"),
            lit("/"),
            col("frame_name")
        )
    )
    .withColumn(
        "Embeddings directory",
        concat(
            lit(f"{embed_root}/"),
            col("ID"),
            lit("/"),
            regexp_replace(col("frame_name"), "\\.jpg$", ""),
            lit("_"),
            col("ID"),
            lit(".pt")
        )
    )
)


# 8) VALIDATE & DROP MISSING

joined = (
    joined
    .withColumn("Frame Directory",      validate_frame(col("Frame Directory")))
    .withColumn("Embeddings directory", validate_embed(col("Embeddings directory")))
    .filter(
        col("Frame Directory").isNotNull() &
        col("Embeddings directory").isNotNull()
    )
)


# 9) RENAME FOR AGGREGATION

joined = joined \
    .withColumnRenamed("Primary Raw",   "orig_primary_raw") \
    .withColumnRenamed("Secondary Raw", "orig_secondary_raw")


# 10) MAJORITY & SECONDARY VOTE PER SECOND

sec_counts = (
    joined.groupBy("sec_ts", "orig_primary_raw").count()
)
rank_window = Window.partitionBy("sec_ts").orderBy(F.desc("count"))
ranked = sec_counts.withColumn("rn", F.row_number().over(rank_window))

primary_per_sec = ranked.filter("rn = 1") \
                       .select("sec_ts", col("orig_primary_raw").alias("Primary Raw"))
secondary_per_sec = ranked.filter("rn = 2") \
                         .select("sec_ts", col("orig_primary_raw").alias("Secondary Raw"))

joined = joined \
    .join(primary_per_sec,   "sec_ts", "left") \
    .join(secondary_per_sec, "sec_ts", "left")


# 11) RECATEGORIZE & FINAL LABEL

behavior_categories = {
    'Buck':'Active Playing','gallop':'Active Playing','Kick':'Active Playing','leap':'Active Playing',
    'leap sideways':'Active Playing','Turn':'Active Playing','Jump':'Active Playing','chasing':'Active Playing',
    'bouncing':'Non Active Playing','head-butting':'Non Active Playing','head-shake':'Non Active Playing',
    'butting fixtures':'Non Active Playing','butting/rubbing straw':'Non Active Playing','mount':'Non Active Playing',
    'frontal pushing':'Non Active Playing','play-bouncing':'Non Active Playing',
    'Individuall':'Not Playing','Milk Feeding':'Not Playing','rest behavior':'Not Playing',
    'out of view':'Not Playing','Management (someone in the stalls)':'Not Playing',
    'Management (someone in the pen)':'Not Playing','(Blanks)':'Not Playing'
}

@udf(StringType())
def categorize(b):
    return behavior_categories.get(b, "Not Playing")

joined = joined \
    .withColumn("Primary Label",   categorize(col("Primary Raw"))) \
    .withColumn("Secondary Label", categorize(col("Secondary Raw"))) \
    .withColumn(
        "Final Label",
        when(
            (col("Primary Label") == "Active Playing") |
            (col("Secondary Label") == "Active Playing"),
            "Active Playing"
        )
        .when(
            (col("Primary Label") == "Non Active Playing") |
            (col("Secondary Label") == "Non Active Playing"),
            "Non Active Playing"
        )
        .otherwise("Not Playing")
    )


# 12) FORMAT TIMESTAMP TO SECOND

joined = joined.withColumn(
    "Timestamp",
    date_format(col("sec_ts"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
)


# 13) FINAL SORT & WRITE

final = joined \
    .orderBy(col("sec_ts"), col("ID"), col("frame_num")) \
    .select(
        "Timestamp",
        "Primary Raw", "Primary Label",
        "Secondary Raw", "Secondary Label",
        "ID", "Final Label",
        "Frame Directory", "Embeddings directory"
    ) \
    .repartition(12)

final.coalesce(1) \
     .write \
     .option("header", True) \
     .mode("overwrite") \
     .csv(output_path)

print(f"Written merged metadata to: {output_path}")